# Module 20: Advanced Python Patterns — Solutions

## Complete Solutions for All Exercises

## Part 1: Decorators — Solutions

In [ ]:
from functools import wraps
import time
from collections import deque

# Exercise 1.1: @log_execution decorator
def log_execution(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = time.time()
        args_str = str(args)[:100] + '...' if len(str(args)) > 100 else str(args)
        print(f'[EXEC] Starting {func.__name__}')
        print(f'[EXEC] Args: {args_str}')
        result = func(*args, **kwargs)
        res_str = str(result)[:100] + '...' if len(str(result)) > 100 else str(result)
        elapsed = time.time() - start
        print(f'[EXEC] {func.__name__} -> {res_str} ({elapsed:.3f}s)')
        return result
    return wrapper

@log_execution
def train_model(data_size, epochs):
    time.sleep(0.1)
    return {'accuracy': 0.95, 'loss': 0.12}

print('=== @log_execution Demo ===')
result = train_model(10000, 10)
print(f'Result: {result}')

In [ ]:
# Exercise 1.2: @rate_limit decorator
def rate_limit(max_calls=5, period=10):
    call_times = deque()
    
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            nonlocal call_times
            now = time.time()
            # Remove old timestamps
            while call_times and call_times[0] < now - period:
                call_times.popleft()
            
            if len(call_times) >= max_calls:
                wait = call_times[0] + period - now
                print(f'[RateLimit] Waiting {wait:.1f}s...')
                time.sleep(wait)
                now = time.time()
                while call_times and call_times[0] < now - period:
                    call_times.popleft()
            
            call_times.append(time.time())
            return func(*args, **kwargs)
        return wrapper
    return decorator

@rate_limit(max_calls=3, period=5)
def predict_api(data):
    return {'prediction': 1, 'confidence': 0.9}

print('=== @rate_limit Demo ===')
for i in range(5):
    result = predict_api(i)
    print(f'  Call {i+1}: {result}')

In [ ]:
# Exercise 1.3: @cache_with_ttl decorator
def cache_with_ttl(ttl_seconds=60):
    cache = {}
    
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            key = str(args) + str(sorted(kwargs.items()))
            now = time.time()
            
            if key in cache:
                value, timestamp = cache[key]
                if now - timestamp < ttl_seconds:
                    print(f'[Cache] HIT for {key[:30]}...')
                    return value
                else:
                    print(f'[Cache] EXPIRED for {key[:30]}...')
                    del cache[key]
            
            print(f'[Cache] MISS for {key[:30]}...')
            result = func(*args, **kwargs)
            cache[key] = (result, time.time())
            return result
        return wrapper
    return decorator

@cache_with_ttl(ttl_seconds=2)
def slow_predict(features):
    time.sleep(0.2)  # Simulate slow model
    return {'prediction': sum(features), 'confidence': 0.9}

print('=== @cache_with_ttl Demo ===')
print('First call (miss):', slow_predict([1, 2, 3]))
print('Second call (hit):', slow_predict([1, 2, 3]))
print('Third call (hit):', slow_predict([1, 2, 3]))
time.sleep(2.5)
print('After TTL (miss):', slow_predict([1, 2, 3]))

## Part 2: Generators — Solutions

In [ ]:
# Exercise 2.1: CSV Streamer Generator
def csv_streamer(file_obj, batch_size=100):
    reader = csv.DictReader(file_obj)
    batch = []
    for row in reader:
        batch.append(row)
        if len(batch) >= batch_size:
            yield batch
            batch = []
    if batch:
        yield batch

# Simulate CSV content
csv_content = 'name,age,score\nAlice,30,95\nBob,25,87\nCharlie,35,92\n' * 50
file_obj = io.StringIO(csv_content)

print('=== CSV Streamer ===')
for i, batch in enumerate(csv_streamer(file_obj, batch_size=30)):
    print(f'  Batch {i+1}: {len(batch)} rows')
print(f'Total batches: {i+1}')

In [ ]:
# Exercise 2.2: Generator Pipeline
def data_source(n):
    for _ in range(n):
        val = random.random() * 10
        yield val if random.random() > 0.1 else None  # 10% None

def clean(stream):
    for item in stream:
        if item is not None and isinstance(item, (int, float)):
            yield item

def normalize(stream):
    items = list(stream)
    if not items:
        return
    min_val, max_val = min(items), max(items)
    range_val = max_val - min_val
    if range_val == 0:
        for item in items:
            yield 0.5
    else:
        for item in items:
            yield (item - min_val) / range_val

def batch(stream, size):
    batch_data = []
    for item in stream:
        batch_data.append(item)
        if len(batch_data) >= size:
            yield batch_data
            batch_data = []
    if batch_data:
        yield batch_data

# Compose pipeline
pipeline = batch(normalize(clean(data_source(1000))), 50)
print('=== Generator Pipeline ===')
for i, batch in enumerate(pipeline):
    if i < 3:
        print(f'  Batch {i+1}: min={min(batch):.3f}, max={max(batch):.3f}, size={len(batch)}')
print(f'Total batches: {i+1}')

## Part 3: Context Managers — Solutions

In [ ]:
# Exercise 3.1: DatabaseSession Context Manager
class DatabaseSession:
    def __init__(self, db_name='ml_db'):
        self.db_name = db_name
        self.connected = False
    
    def __enter__(self):
        print(f'[DB] Connecting to {self.db_name}...')
        time.sleep(0.05)  # Simulate connection
        self.connected = True
        self.transaction_active = True
        print('[DB] Connected')
        return self
    
    def __exit__(self, exc_type, exc_val, exc_tb):
        if exc_type is not None:
            print(f'[DB] Rollback due to: {exc_val}')
        else:
            print('[DB] Commit')
        self.connected = False
        self.transaction_active = False
        print('[DB] Disconnected')
        return False  # Don't suppress exceptions
    
    def query(self, sql):
        if not self.connected:
            raise RuntimeError('Not connected')
        return f'Results for: {sql}'

print('=== DatabaseSession Demo ===')
with DatabaseSession() as db:
    result = db.query('SELECT * FROM features')
    print(f'  {result}')

print('\nWith exception:')
try:
    with DatabaseSession() as db:
        raise ValueError('Bad data')
except ValueError:
    print('  Exception handled, session rolled back')

In [ ]:
from contextlib import contextmanager
from concurrent.futures import ProcessPoolExecutor

# Exercise 3.2: WorkerPool Context Manager
@contextmanager
def worker_pool(n_workers=4):
    print(f'[Pool] Creating pool with {n_workers} workers...')
    start = time.time()
    pool = ProcessPoolExecutor(max_workers=n_workers)
    try:
        yield pool
    finally:
        pool.shutdown()
        elapsed = time.time() - start
        print(f'[Pool] Shutdown complete. Pool active for {elapsed:.2f}s')

def square(x):
    return x * x

print('=== WorkerPool Demo ===')
with worker_pool(4) as pool:
    results = list(pool.map(square, range(20)))
    print(f'  Results: {results[:5]}...{results[-5:]}')

## Part 4: Design Patterns — Solutions

In [ ]:
import threading

# Exercise 4.1: Thread-safe Singleton FeatureStore
class FeatureStore:
    _instance = None
    _lock = threading.Lock()
    
    def __new__(cls):
        if cls._instance is None:
            with cls._lock:
                if cls._instance is None:  # Double check
                    cls._instance = super().__new__(cls)
                    cls._instance._features = {}
                    cls._instance._store_lock = threading.Lock()
        return cls._instance
    
    def set_feature(self, name, version, data):
        with self._store_lock:
            key = f'{name}:{version}'
            self._features[key] = data
            print(f'[FeatureStore] Stored: {key}')
    
    def get_feature(self, name, version):
        with self._store_lock:
            return self._features.get(f'{name}:{version}')
    
    def list_features(self):
        with self._store_lock:
            return list(self._features.keys())

fs1 = FeatureStore()
fs2 = FeatureStore()
print(f'Singleton check: {fs1 is fs2}')
fs1.set_feature('user_age', 'v1', [25, 30, 35])
fs1.set_feature('user_income', 'v1', [50000, 60000, 70000])
print(f'Features: {fs2.list_features()}')

In [ ]:
# Exercise 4.2: DataLoaderFactory
from abc import ABC, abstractmethod

class DataLoader(ABC):
    @abstractmethod
    def load(self, path):
        pass

class CSVDataLoader(DataLoader):
    def load(self, path):
        print(f'[CSVLoader] Loading {path}')
        return {'format': 'csv', 'rows': 1000, 'path': path}

class ParquetDataLoader(DataLoader):
    def load(self, path):
        print(f'[ParquetLoader] Loading {path}')
        return {'format': 'parquet', 'rows': 5000, 'path': path}

class SQLDataLoader(DataLoader):
    def load(self, path):
        print(f'[SQLLoader] Loading from {path}')
        return {'format': 'sql', 'rows': 10000, 'path': path}

class DataLoaderFactory:
    @staticmethod
    def create(path):
        if path.endswith('.csv'):
            return CSVDataLoader()
        elif path.endswith('.parquet'):
            return ParquetDataLoader()
        elif path.startswith('sqlite://') or path.startswith('postgres://'):
            return SQLDataLoader()
        raise ValueError(f'Unknown format for: {path}')

print('=== DataLoaderFactory ===')
for path in ['data.csv', 'features.parquet', 'sqlite:///ml.db']:
    loader = DataLoaderFactory.create(path)
    data = loader.load(path)
    print(f'  Loaded: {data}')

In [ ]:
# Exercise 4.3: Training Observer Pattern
class TrainingSubject:
    def __init__(self):
        self._observers = []
        self._stop = False
    
    def attach(self, observer):
        self._observers.append(observer)
    
    def detach(self, observer):
        self._observers.remove(observer)
    
    def notify(self, event_type, data):
        for obs in self._observers:
            obs.update(event_type, data)
            if getattr(obs, 'should_stop', False):
                self._stop = True
    
    def train(self, n_epochs=10):
        for epoch in range(n_epochs):
            if self._stop:
                print(f'[Training] Early stopping at epoch {epoch}')
                break
            loss = 1.0 / (epoch + 1)
            accuracy = 1 - loss
            self.notify('epoch', {'epoch': epoch, 'loss': loss, 'accuracy': accuracy})
            time.sleep(0.05)

class ProgressBarObserver:
    def update(self, event_type, data):
        if event_type == 'epoch':
            bar_len = 20
            filled = int(bar_len * data['accuracy'])
            bar = '#' * filled + '-' * (bar_len - filled)
            print(f'  Epoch {data["epoch"]:2d}: [{bar}] loss={data["loss"]:.4f}')

class EarlyStoppingObserver:
    def __init__(self, patience=3):
        self.patience = patience
        self.best_loss = float('inf')
        self.counter = 0
        self.should_stop = False
    
    def update(self, event_type, data):
        if event_type == 'epoch':
            loss = data['loss']
            if loss < self.best_loss:
                self.best_loss = loss
                self.counter = 0
            else:
                self.counter += 1
                if self.counter >= self.patience:
                    self.should_stop = True

print('=== Training Observer ===')
trainer = TrainingSubject()
trainer.attach(ProgressBarObserver())
trainer.attach(EarlyStoppingObserver(patience=2))
trainer.train(n_epochs=10)

## Part 5: Dependency Injection — Solutions

In [ ]:
from abc import ABC, abstractmethod

# Interfaces
class ModelLoader(ABC):
    @abstractmethod
    def load(self, version):
        pass

class Preprocessor(ABC):
    @abstractmethod
    def preprocess(self, data):
        pass

class Logger(ABC):
    @abstractmethod
    def log(self, message):
        pass

# Real implementations
class ProductionModelLoader(ModelLoader):
    def load(self, version):
        print(f'[Prod] Loading model v{version}...')
        return {'name': 'model', 'version': version}

class ProductionPreprocessor(Preprocessor):
    def preprocess(self, data):
        print(f'[Prod] Preprocessing {len(data)} items...')
        return [x * 2 for x in data]

class ProductionLogger(Logger):
    def log(self, message):
        print(f'[Prod Log] {message}')

# Mock implementations for testing
class MockModelLoader(ModelLoader):
    def load(self, version):
        return {'name': 'mock', 'version': version}

class MockPreprocessor(Preprocessor):
    def preprocess(self, data):
        return [1, 2, 3]

class MockLogger(Logger):
    def __init__(self):
        self.messages = []
    def log(self, message):
        self.messages.append(message)

# PredictionService with DI
class PredictionService:
    def __init__(self, model_loader, preprocessor, logger):
        self.model_loader = model_loader
        self.preprocessor = preprocessor
        self.logger = logger
        self.model = None
    
    def initialize(self, version):
        self.model = self.model_loader.load(version)
        self.logger.log(f'Initialized model v{version}')
    
    def predict(self, data):
        processed = self.preprocessor.preprocess(data)
        self.logger.log(f'Predicted {len(data)} samples')
        return [sum(processed)]

print('=== DI PredictionService (Production) ===')
prod = PredictionService(ProductionModelLoader(), ProductionPreprocessor(), ProductionLogger())
prod.initialize('2.1')
pred = prod.predict([1, 2, 3])
print(f'  Prediction: {pred}')

print('\n=== DI PredictionService (Test with Mocks) ===')
mock_logger = MockLogger()
test = PredictionService(MockModelLoader(), MockPreprocessor(), mock_logger)
test.initialize('test')
pred = test.predict([10, 20, 30])
print(f'  Prediction: {pred}')
print(f'  Log messages: {mock_logger.messages}')